In [ ]:
pip install pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 69.7 MB/s eta 0:00:00


In [ ]:
pip install pynini

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.5/165.5 MB 4.6 MB/s eta 0:00:00


In [ ]:
# Ячейка 1: pymorphy3 backend (VerbProcessor)
from dataclasses import dataclass, field
from typing import Dict, Optional, Tuple, Any
import sys
import time

# Попытка импортировать pymorphy3 — если в окружении нет, не падаем при импорте модуля.
try:
    import pymorphy3
except Exception:
    pymorphy3 = None

@dataclass
class VerbForm:
    infinitive: str
    tense: str       # 'past', 'present', 'future'
    aspect: str      # 'perf', 'impf', 'biaspectual'
    person: Optional[str] = None  # '1sg','2sg','3sg','1pl','2pl','3pl'
    gender: Optional[str] = None  # 'masc','femn','neut'
    number: Optional[str] = None  # 'sing','plur'
    form: str = ""
    _meta: Dict[str, Any] = field(default_factory=dict)

class VerbProcessor:

    def __init__(self):
        if not pymorphy3:
            raise RuntimeError("pymorphy3 не установлен в окружении. Установите pymorphy3 или используйте fst-режим.")
        self.morph = pymorphy3.MorphAnalyzer()

    def _determine_aspect(self, parsing, verb: str) -> str:
        aspect_exceptions = {
            'бежать': 'impf', 'бегать': 'impf', 'летать': 'impf', 'бросить': 'perf'
        }
        if verb in aspect_exceptions:
            return aspect_exceptions[verb]
        return 'perf' if 'perf' in parsing.tag else 'impf'

    def _parse_verb(self, word: str):
        parsings = self.morph.parse(word)
        for parsing in parsings:
            if 'VERB' in parsing.tag or 'INFN' in parsing.tag:
                return parsing
        return None

    def _extract_grammemes(self, tag, aspect: str = None) -> Dict[str, str]:
        grammemes = {}
        try:
            # время
            if 'past' in tag:
                grammemes['tense'] = 'past'
            elif 'pres' in tag:
                # для совершенных глаголов 'pres' интерпретируем как future
                if aspect == 'perf' and aspect != 'biaspectual':
                    grammemes['tense'] = 'future'
                else:
                    grammemes['tense'] = 'present'
            elif 'futr' in tag:
                grammemes['tense'] = 'future'

            if 'tense' not in grammemes:
                return grammemes

            # вид
            grammemes['aspect'] = 'perf' if 'perf' in tag else 'impf'

            # лицо
            if '1per' in tag:
                grammemes['person'] = '1sg' if 'sing' in tag else '1pl'
            elif '2per' in tag:
                grammemes['person'] = '2sg' if 'sing' in tag else '2pl'
            elif '3per' in tag:
                grammemes['person'] = '3sg' if 'sing' in tag else '3pl'

            # род (для прошедшего)
            if 'masc' in tag:
                grammemes['gender'] = 'masc'
            elif 'femn' in tag:
                grammemes['gender'] = 'femn'
            elif 'neut' in tag:
                grammemes['gender'] = 'neut'

            # число
            grammemes['number'] = 'sing' if 'sing' in tag else 'plur'

        except Exception as e:
            print(f" Ошибка при разборе тега: {e}")
        return grammemes

    def analyze(self, form: str) -> Optional[VerbForm]:
        start = time.time()
        # сложное будущее (буду читать)
        future_prefixes = {
            'буду': '1sg', 'будешь': '2sg', 'будет': '3sg',
            'будем': '1pl', 'будете': '2pl', 'будут': '3pl'
        }
        form = form.strip().lower()
        for prefix, person in future_prefixes.items():
            if form.startswith(prefix + ' '):
                verb_part = form[len(prefix) + 1:]
                verb_parsing = self._parse_verb(verb_part)
                if verb_parsing:
                    synthesized = self.synthesize(verb_parsing.normal_form)
                    if synthesized:
                        aspect_from_synth = synthesized['infinitive']['вид']
                        if aspect_from_synth == 'двувидовой':
                            aspect = 'biaspectual'
                        elif aspect_from_synth == 'совершенный':
                            aspect = 'perf'
                        else:
                            aspect = 'impf'
                    else:
                        aspect = 'impf'
                    vf = VerbForm(infinitive=verb_parsing.normal_form, tense='future', aspect=aspect,
                                  person=person, number='sing' if person.endswith('sg') else 'plur', form=form)
                    vf._meta['elapsed_seconds'] = time.time() - start
                    return vf

        parsing = self._parse_verb(form)
        if not parsing:
            return None
        infinitive = parsing.normal_form
        synthesized = self.synthesize(infinitive)
        # определяем аспект на основе синтеза
        if synthesized:
            aspect_from_synth = synthesized['infinitive']['вид']
            if aspect_from_synth == 'двувидовой':
                aspect = 'biaspectual'
            elif aspect_from_synth == 'совершенный':
                aspect = 'perf'
            else:
                aspect = 'impf'
        else:
            aspect = 'perf' if 'perf' in parsing.tag else 'impf'

        grammemes = self._extract_grammemes(parsing.tag, aspect)
        grammemes['aspect'] = aspect
        if 'tense' not in grammemes:
            return None
        vf = VerbForm(infinitive=infinitive, form=form, **grammemes)
        vf._meta['elapsed_seconds'] = time.time() - start
        return vf

    def synthesize(self, verb: str) -> Dict[str, Dict]:
        start = time.time()
        all_parsings = self.morph.parse(verb)
        best_parsing = None
        for p in all_parsings:
            if 'VERB' in p.tag or 'INFN' in p.tag:
                if p.normal_form == verb:
                    best_parsing = p
                    break
                elif best_parsing is None:
                    best_parsing = p
        if not best_parsing:
            return {}

        aspect = 'perf' if 'perf' in best_parsing.tag else 'impf'
        result = {
            'infinitive': {'форма': verb, 'вид': 'совершенный' if aspect == 'perf' else 'несовершенный'},
            'past': {}, 'present': {}, 'future_simple': {}, 'future_compound': {}
        }
        # генерируем формы из лексемы
        all_forms = best_parsing.lexeme
        for form in all_forms:
            if not self._is_personal_verb_form(form):
                continue
            form_grammemes = self._extract_grammemes(form.tag, aspect)
            if 'tense' not in form_grammemes:
                continue
            self._add_form_to_result(result, form.word, form_grammemes, aspect)

        # проверка двувидовости
        is_biaspectual = self._is_likely_biaspectual(best_parsing, verb, result)
        if is_biaspectual:
            result['infinitive']['вид'] = 'двувидовой'
            aspect = 'biaspectual'


        if aspect == 'perf' and not is_biaspectual:

            if result.get('present'):
                result['future_simple'] = result['present'].copy()
                result['present'] = {}
        elif aspect == 'impf':

            result['future_simple'] = {}
        elif is_biaspectual:

            if result.get('future_simple') and not result.get('present'):
                result['present'] = result['future_simple'].copy()
            elif result.get('present') and not result.get('future_simple'):
                result['future_simple'] = result['present'].copy()
            result['future_simple'] = {}


        if (aspect == 'impf' or is_biaspectual):
            self._add_compound_future(result, verb)

        result['_meta'] = {'model': 'pymorphy3', 'elapsed_seconds': time.time() - start}
        return result

    def _add_form_to_result(self, result, form_word, grammemes, aspect):
        tense = grammemes['tense']
        if tense == 'past':
            gender = grammemes.get('gender')
            if gender:
                key = {'masc': 'мужской род', 'femn': 'женский род', 'neut': 'средний род'}[gender]
                result['past'][key] = form_word
            else:
                result['past']['множественное число'] = form_word
        elif tense == 'present':
            person_map = {'1sg': 'я', '2sg': 'ты', '3sg': 'он/она', '1pl': 'мы', '2pl': 'вы', '3pl': 'они'}
            if grammemes.get('person') in person_map:
                result['present'][person_map[grammemes['person']]] = form_word
        elif tense == 'future':
            person_map = {'1sg': 'я', '2sg': 'ты', '3sg': 'он/она', '1pl': 'мы', '2pl': 'вы', '3pl': 'они'}
            if grammemes.get('person') in person_map:
                result['future_simple'][person_map[grammemes['person']]] = form_word

    def _is_personal_verb_form(self, form) -> bool:

        if 'PRTF' in form.tag or 'GRND' in form.tag:
            return False
        if 'Arch' in form.tag:
            return False
        if self._is_obsolete_form(form):
            return False
        return 'VERB' in form.tag

    def _is_obsolete_form(self, form) -> bool:
        word = form.word.lower()
        infinitive = form.normal_form
        obsolete_forms = {
            'хотеть': ['хошь','хочим','хочем', 'хочите', 'хочут','хочете'],
            'смотреть': ['смотрют'],
            'бежать': ['бежу', 'бежат'],
            'слышать': ['слышут'],
            'победить': ['побежу'],
            'убедить':['убежу'],
            'дерзить':['держу'],
            'ощутить':['ощущу'],
            'пронзить':['пронжу'],
            'чудить':['чужу'],
            'дудеть':['дудю'],
            'бузеть':['бузю'],
            'дышать':['дышут'],
            'держать':['держут'],
            'гнать':['гонют'],
            'видеть':['видют'],
            'сыпать':['сыпаю','сыпаешь','сыпаем','сыпает','сыпаете','сыпают']
        }
        if infinitive in obsolete_forms and word in obsolete_forms[infinitive]:
            return True
        return False

    def _add_compound_future(self, result, verb):
        future_forms = {
            'я': f'буду {verb}', 'ты': f'будешь {verb}', 'он/она': f'будет {verb}',
            'мы': f'будем {verb}', 'вы': f'будете {verb}', 'они': f'будут {verb}'
        }
        result['future_compound'] = future_forms

    def _is_likely_biaspectual(self, parsing, verb: str, forms: dict = None) -> bool:

        if forms and self._has_matching_present_future_forms(forms):
            return True
        if self._has_biaspectual_morphology(parsing, verb):
            return True
        return False

    def _has_matching_present_future_forms(self, forms: dict) -> bool:
        present = forms.get('present', {})
        future = forms.get('future_simple', {})
        if present and future:
            for person in ['я', 'ты', 'он/она', 'мы', 'вы', 'они']:
                if person in present and person in future and present[person] != future[person]:
                    return False
            return True
        return False

    def _has_biaspectual_morphology(self, parsing, verb: str) -> bool:
        if 'perf' in parsing.tag and 'impf' in parsing.tag:
            return True
        has_present = any('pres' in form.tag for form in parsing.lexeme)
        if has_present and 'perf' in parsing.tag:
            return True
        all_parsings = self.morph.parse(verb)
        aspects = set()
        for p in all_parsings:
            if 'VERB' in p.tag or 'INFN' in p.tag:
                if 'perf' in p.tag: aspects.add('perf')
                if 'impf' in p.tag: aspects.add('impf')
        return len(aspects) > 1


In [ ]:
# Ячейка 2: SimpleFST + константы
from typing import Dict, Tuple, Optional
import pynini
from pynini.lib import rewrite
import time

# Простые регулярности/исключения для образования основ
ALTERNATIONS = {'пис': 'пиш', 'сп': 'спл', 'бег': 'беж', 'нос': 'нош', 'ид': 'ид',
                'жд': 'жд','люб': 'любл', 'смотр': 'смотр', 'слыш': 'слыш', 'вид': 'виж',
                'учи': 'уч', 'стро': 'стро', 'ход': 'хож',
                'мог': 'мож', 'гн': 'гон', 'кла': 'клад', 'дост': 'достан',
                }

STEM_EXCEPTIONS = {
    'писать': 'пиш', 'спать': 'спл', 'идти': 'ид', 'ехать': 'ед', 'бежать': 'беж',
    'лежать': 'леж', 'стоять': 'сто', 'сидеть': 'сид', 'держать': 'держ', 'смотреть': 'смотр',

    'делать': 'дела',
    'быть': 'бы',
    'иметь': 'име',
    'сказать': 'скаж',
    'говорить': 'говор',
    'видеть': 'виж',
    'стоять': 'сто',
    'думать': 'дума',
    'знать': 'зна',
    'понимать': 'понима',
    'работать': 'работа',
    'жить': 'жив',
    'читать': 'чита',
    'любить': 'любл',
    'ненавидеть': 'ненавид',
    'играть': 'игра',
    'взять': 'вз',
    'купить': 'куп',
    'покупать': 'покупа',
    'продавать': 'прода',
    'учиться': 'уч',
    'учить': 'учу',
    'помнить': 'помни',
    'забыть': 'заб',
    'ждать': 'жд',
    'ходить': 'хож',
    'приходить': 'приход',
    'уходить': 'уход',
    'слышать': 'слыш',
    'написать': 'напиш',
    'прочитать': 'прочита',
    'встать': 'встан',
    'лечь': 'леч',
    'начать': 'нач',
    'закончить': 'законч',
    'выйти': 'выйд',
    'войти': 'войд',
    'вздохнуть': 'вздох',
    'улыбнуться': 'улыбн',
    'плакать': 'пла',
    'смеяться': 'сме',
    'уметь': 'ум',
    'мочь': 'мож',
    'решить': 'реш',
    'проверить': 'провер',
    'искать': 'ищ',
    'найти': 'найд',
    'встретить': 'встрет',
    'спросить': 'спрос',
    'ответить': 'ответ',
    'объяснить': 'объясн',
    'понять': 'пойм',
    'хотеть': 'хоч',
    'пойти': 'пойд',
    'прийти': 'прийд',
    'принести': 'принес',
    'нести': 'нес',
    'посмотреть': 'посмотр',
    'послушать': 'послуш',
    'танцевать': 'танц',
    'петь': 'пою',
    'рисовать': 'рисова',
    'готовить': 'готов',
    'открыть': 'откро',
    'закрыть': 'закро',
    'позвонить': 'позвон',
    'достать': 'достан',
    'класть': 'клад',
    'вести': 'вед',
    'плыть': 'плы',
    'лететь': 'лет',
    'летать': 'лета',
    'дудеть': 'дуд',
    'улыбаться': 'улыб',
    'бояться': 'бо',
    'надеяться': 'наде',
    'казаться': 'каж',
    'пользоваться': 'польз',
    'заниматься': 'заним',
    'приближаться': 'приближ',
    'отдаляться': 'отдал',
    'встречаться': 'встреч',
    'общаться': 'общ',
    'прятаться': 'прят',
    'сдаваться': 'сда',
    'двигаться': 'двиг',
    'перемещаться': 'перемещ',
    'находиться': 'наход',
    'прощаться': 'прощ',
    'развиваться': 'развив',
    'мыться': 'мы',
    'бриться': 'бр',
    'одеваться': 'одева',
    'раздеваться': 'раздева',
    'кушаться': 'куша',
    'собираться': 'собира',
    'жениться': 'жен',
    'увольняться': 'увольн',
    'здороваться': 'здорова',
    'извиняться': 'извин',
    'радоваться': 'раду',
    'удивляться': 'удив',
    'обниматься': 'обнима',
    'целоваться': 'целова',
    'знакомиться': 'знаком',
    'переписываться': 'перепис',
    'ссориться': 'ссор',
    'мириться': 'мир',
    'интересоваться': 'интерес',
    'называться': 'называ',
    'остановиться': 'останов',
    'продаваться': 'прода',
    'организовать': 'организ',
    'атаковать': 'атак',
    'исследовать': 'исследова',
    'строить': 'стро',
    'построить': 'постро',
    'ломать': 'лома',
    'сломать': 'слом',
    'брать': 'бер',
    'давать': 'дава',
    'кричать': 'крича',
    'молчать': 'молча',
    'помогать': 'помога',
    'помочь': 'помог',
    'мешать': 'меша',
    'помешать': 'помеш',
    'приготовить': 'приготов',
    'начинать': 'начина',
    'заканчивать': 'заканчива',
    'решать': 'реша',
    'забывать': 'забыва',
    'проверять': 'проверя',
    'звонить': 'звон',
    'проверить': 'провер',
    'позвонить': 'позвон',
    'увидеть': 'увид',
    'услышать': 'услыш',
    'подождать': 'подожд',
    'прибежать': 'прибеж',
    'приносить': 'принос',
    'положить': 'полож',
    'сохранять': 'сохраня',
    'передавать': 'передава',
    'получать': 'получа',
    'выбирать': 'выбира',
    'поддерживать': 'поддержива',
    'обслуживать': 'обслужива',
    'ждать': 'жд',
    'смотреть': 'смотр',
    'любить': 'люб',
    'слышать': 'слыш',
    'видеть': 'вид',
    'ненавидеть': 'ненавид',
    'гнать': 'гон',
    'держать': 'держ',
    'дышать': 'дыш',
    'терпеть': 'терп',
    'зависеть': 'завис',
    'вертеть': 'верт',
    'брить': 'бре',
    'стелить': 'стел',
    'учить': 'уч',
    'строить': 'стро',
    'ходить': 'ход',
    'класть': 'клад',
    'стоять': 'сто',
    'лежать': 'леж',
    'быть': 'буд',
}

# ЯВНЫЕ СПИСКИ ДЛЯ КОРРЕКТНОГО ОПРЕДЕЛЕНИЯ ВИДА
PERFECTIVE_VERBS = {
    "сказать","прочитать","написать","купить","взять","встать","выйти","войти",
    "принести","положить","помочь","прийти","уйти","построить","сломать","закончить",
    "начать","приготовить","проверить","позвонить","закричать","замолчать",
    "прочитать","написать","построить","сломать","положить","упасть","достать","помочь","принести","купить",

}
IMPERFECTIVE_VERBS = {
    "делать", "ждать", "смотреть", "любить", "слышать", "видеть", "ненавидеть", "гнать", "держать", "дышать",
    "терпеть", "зависеть", "вертеть", "брить", "стелить", "учить", "строить", "ходить", "стоять", "лежать",
    "говорить", "думать", "знать", "понимать", "работать", "жить", "сидеть", "лежать", "петь", "танцевать",
    "плавать", "бегать", "ездить", "спрашивать", "отвечать", "смеяться", "учиться", "бояться", "надеяться",
    "находиться", "радоваться", "пользоваться", "мыться", "бриться", "общаться", "учить", "спать",
}


BIA_VERBS = {
    "организовать", "исследовать"
}

# ЯВНЫЕ (EXPLICIT) ФОРМЫ:
EXPLICIT_VERB_FORMS: Dict[str, Dict[str, Dict[str, str]]] = {

    'есть': {
        'present': {'я': 'ем', 'ты': 'ешь', 'он/она': 'ест', 'мы': 'едим', 'вы': 'едите', 'они': 'едят'},
        'past': {'мужской род': 'ел', 'женский род': 'ела', 'средний род': 'ело', 'множественное число': 'ели'}
    },
    'пить': {
        'present': {'я': 'пью', 'ты': 'пьёшь', 'он/она': 'пьёт', 'мы': 'пьём', 'вы': 'пьёте', 'они': 'пьют'},
        'past': {'мужской род': 'пил', 'женский род': 'пила', 'средний род': 'пило', 'множественное число': 'пили'}
    },
    'дать': {
        'present': {'я': 'дам', 'ты': 'дашь', 'он/она': 'даст', 'мы': 'дадим', 'вы': 'дадите', 'они': 'дадут'},
        'past': {'мужской род': 'дал', 'женский род': 'дала', 'средний род': 'дало', 'множественное число': 'дали'}
    },
    'брать': {
        'present': {'я': 'беру', 'ты': 'берёшь', 'он/она': 'берёт', 'мы': 'берём', 'вы': 'берёте', 'они': 'берут'},
        'past': {'мужской род': 'брал', 'женский род': 'брала', 'средний род': 'брало', 'множественное число': 'брали'}
    },
    'взять': {
        'future_simple': {'я': 'возьму', 'ты': 'возьмёшь', 'он/она': 'возьмёт', 'мы': 'возьмём', 'вы': 'возьмёте', 'они': 'возьмут'},
        'past': {'мужской род': 'взял', 'женский род': 'взяла', 'средний род': 'взяло', 'множественное число': 'взяли'}
    },
    'лежать': {
        'present': {'я': 'лежу', 'ты': 'лежишь', 'он/она': 'лежит', 'мы': 'лежим', 'вы': 'лежите', 'они': 'лежат'},
        'past': {'мужской род': 'лежал', 'женский род': 'лежала', 'средний род': 'лежало', 'множественное число': 'лежали'}
    },
    'встать': {
        'future_simple': {'я': 'встану', 'ты': 'встанешь', 'он/она': 'встанет', 'мы': 'встанем', 'вы': 'встанете', 'они': 'встанут'},
        'past': {'мужской род': 'встал', 'женский род': 'встала', 'средний род': 'встало', 'множественное число': 'встали'}
    },
    'спать': {
        'present': {'я': 'сплю', 'ты': 'спишь', 'он/она': 'спит', 'мы': 'спим', 'вы': 'спите', 'они': 'спят'},
        'past': {'мужской род': 'спал', 'женский род': 'спала', 'средний род': 'спало', 'множественное число': 'спали'}
    },


    'смотреть': {
        'present': {'я': 'смотрю', 'ты': 'смотришь', 'он/она': 'смотрит', 'мы': 'смотрим', 'вы': 'смотрите', 'они': 'смотрят'},
        'past': {'мужской род': 'смотрел', 'женский род': 'смотрела', 'средний род': 'смотрело', 'множественное число': 'смотрели'}
    },
    'любить': {
        'present': {'я': 'люблю', 'ты': 'любишь', 'он/она': 'любит', 'мы': 'любим', 'вы': 'любите', 'они': 'любят'},
        'past': {'мужской род': 'любил', 'женский род': 'любила', 'средний род': 'любило', 'множественное число': 'любили'}
    },
    'слышать': {
        'present': {'я': 'слышу', 'ты': 'слышишь', 'он/она': 'слышит', 'мы': 'слышим', 'вы': 'слышите', 'они': 'слышат'},
        'past': {'мужской род': 'слышал', 'женский род': 'слышала', 'средний род': 'слышало', 'множественное число': 'слышали'}
    },
    'видеть': {
        'present': {'я': 'вижу', 'ты': 'видишь', 'он/она': 'видит', 'мы': 'видим', 'вы': 'видите', 'они': 'видят'},
        'past': {'мужской род': 'видел', 'женский род': 'видела', 'средний род': 'видело', 'множественное число': 'видели'}
    },
    'ненавидеть': {
        'present': {'я': 'ненавижу', 'ты': 'ненавидишь', 'он/она': 'ненавидит', 'мы': 'ненавидим', 'вы': 'ненавидите', 'они': 'ненавидят'},
        'past': {'мужской род': 'ненавидел', 'женский род': 'ненавидела', 'средний род': 'ненавидело', 'множественное число': 'ненавидели'}
    },
    'держать': {
        'present': {'я': 'держу', 'ты': 'держишь', 'он/она': 'держит', 'мы': 'держим', 'вы': 'держите', 'они': 'держат'},
        'past': {'мужской род': 'держал', 'женский род': 'держала', 'средний род': 'держало', 'множественное число': 'держали'}
    },
    'брить': {
        'present': {'я': 'брею', 'ты': 'бреешь', 'он/она': 'бреет', 'мы': 'бреем', 'вы': 'бреете', 'они': 'бреют'},
        'past': {'мужской род': 'брил', 'женский род': 'брила', 'средний род': 'брило', 'множественное число': 'брили'}
    },
    'стелить': {
        'present': {'я': 'стелю', 'ты': 'стелешь', 'он/она': 'стелет', 'мы': 'стелем', 'вы': 'стелете', 'они': 'стелют'},
        'past': {'мужской род': 'стелил', 'женский род': 'стелила', 'средний род': 'стелило', 'множественное число': 'стелили'}
    },
    'учить': {
        'present': {'я': 'учу', 'ты': 'учишь', 'он/она': 'учит', 'мы': 'учим', 'вы': 'учите', 'они': 'учат'},
        'past': {'мужской род': 'учил', 'женский род': 'учила', 'средний род': 'учило', 'множественное число': 'учили'}
    },
    'строить': {
        'present': {'я': 'строю', 'ты': 'строишь', 'он/она': 'строит', 'мы': 'строим', 'вы': 'строите', 'они': 'строят'},
        'past': {'мужской род': 'строил', 'женский род': 'строила', 'средний род': 'строило', 'множественное число': 'строили'}
    },
    'ходить': {
        'present': {'я': 'хожу', 'ты': 'ходишь', 'он/она': 'ходит', 'мы': 'ходим', 'вы': 'ходите', 'они': 'ходят'},
        'past': {'мужской род': 'ходил', 'женский род': 'ходила', 'средний род': 'ходило', 'множественное число': 'ходили'}
    },
    'написать': {
        'future_simple': {'я': 'напишу', 'ты': 'напишешь', 'он/она': 'напишет', 'мы': 'напишем', 'вы': 'напишете', 'они': 'напишут'},
        'past': {'мужской род': 'написал', 'женский род': 'написала', 'средний род': 'написало', 'множественное число': 'написали'}
    },
    'прочитать': {
        'future_simple': {'я': 'прочитаю', 'ты': 'прочитаешь', 'он/она': 'прочитает', 'мы': 'прочитаем', 'вы': 'прочитаете', 'они': 'прочитают'},
        'past': {'мужской род': 'прочитал', 'женский род': 'прочитала', 'средний род': 'прочитало', 'множественное число': 'прочитали'}
    },
    'купить': {
        'future_simple': {'я': 'куплю', 'ты': 'купишь', 'он/она': 'купит', 'мы': 'купим', 'вы': 'купите', 'они': 'купят'},
        'past': {'мужской род': 'купил', 'женский род': 'купила', 'средний род': 'купило', 'множественное число': 'купили'}
    },
    'помочь': {
        'future_simple': {'я': 'помогу', 'ты': 'поможешь', 'он/она': 'поможет', 'мы': 'поможем', 'вы': 'поможете', 'они': 'помогут'},
        'past': {'мужской род': 'помог', 'женский род': 'помогла', 'средний род': 'помогло', 'множественное число': 'помогли'}
    },
    'достать': {
        'future_simple': {'я': 'достану', 'ты': 'достанешь', 'он/она': 'достанет', 'мы': 'достанем', 'вы': 'достанете', 'они': 'достанут'},
        'past': {'мужской род': 'достал', 'женский род': 'достала', 'средний род': 'достало', 'множественное число': 'достали'}
    },
    'принести': {
        'future_simple': {'я': 'принесу', 'ты': 'принесёшь', 'он/она': 'принесёт', 'мы': 'принесём', 'вы': 'принесёте', 'они': 'принесут'},
        'past': {'мужской род': 'принёс', 'женский род': 'принесла', 'средний род': 'принесло', 'множественное число': 'принесли'}
    },
    'класть': {
        'present': {'я': 'кладу', 'ты': 'кладёшь', 'он/она': 'кладёт', 'мы': 'кладём', 'вы': 'кладёте', 'они': 'кладут'},
        'past': {'мужской род': 'клад', 'женский род': 'кладела', 'средний род': 'кладло', 'множественное число': 'кладли'}
    },
    'мочь': {
        'present': {'я': 'могу', 'ты': 'можешь', 'он/она': 'может', 'мы': 'можем', 'вы': 'можете', 'они': 'могут'},
        'past': {'мужской род': 'мог', 'женский род': 'могла', 'средний род': 'могло', 'множественное число': 'могли'}
    },
    'стоять': {
        'present': {'я': 'стою', 'ты': 'стоишь', 'он/она': 'стоит', 'мы': 'стоим', 'вы': 'стоите', 'они': 'стоят'},
        'past': {'мужской род': 'стоял', 'женский род': 'стояла', 'средний род': 'стояло', 'множественное число': 'стояли'}
    },
    'идти': {
        'present': {'я': 'иду', 'ты': 'идёшь', 'он/она': 'идёт', 'мы': 'идём', 'вы': 'идёте', 'они': 'идут'},
        'past': {'мужской род': 'шёл', 'женский род': 'шла', 'средний род': 'шло', 'множественное число': 'шли'}
    },
    'быть': {
        'present': {'я': 'буду', 'ты': 'будешь', 'он/она': 'будет',
                    'мы': 'будем', 'вы': 'будете', 'они': 'будут'},
        'past': {'мужской род': 'был', 'женский род': 'была',
                 'средний род': 'было', 'множественное число': 'были'}
    },

    'гнать': {
        'present': {'я': 'гоню', 'ты': 'гонишь', 'он/она': 'гонит',
                    'мы': 'гоним', 'вы': 'гоните', 'они': 'гонят'}
    },

    'начать': {
        'future_simple': {
            'я': 'начну', 'ты': 'начнёшь', 'он/она': 'начнёт',
            'мы': 'начнём', 'вы': 'начнёте', 'они': 'начнут'
        },
        'past': {
            'мужской род': 'начал', 'женский род': 'начала',
            'средний род': 'начало', 'множественное число': 'начали'
        }
    },

    'смеяться': {
        'present': {
            'я': 'смеюсь', 'ты': 'смеёшься', 'он/она': 'смеётся',
            'мы': 'смеёмся', 'вы': 'смеётесь', 'они': 'смеются'
        },
        'past': {
            'мужской род': 'смеялся', 'женский род': 'смеялась',
            'средний род': 'смеялось', 'множественное число': 'смеялись'
        }
    },

    'бояться': {
        'present': {
            'я': 'боюсь', 'ты': 'боишься', 'он/она': 'боится',
            'мы': 'боимся', 'вы': 'боитесь', 'они': 'боятся'
        },
        'past': {
            'мужской род': 'боялся', 'женский род': 'боялась',
            'средний род': 'боялось', 'множественное число': 'боялись'
        }
    },

    'жениться': {
        'present': {
            'я': 'женюсь', 'ты': 'женишься', 'он/она': 'женится',
            'мы': 'женимся', 'вы': 'женитесь', 'они': 'женятся'
        },
        'past': {
            'мужской род': 'женился', 'женский род': 'женилась',
            'средний род': 'женилось', 'множественное число': 'женились'
        }
    },

    'организовать': {
        'present': {
            'я': 'организую', 'ты': 'организуешь', 'он/она': 'организует',
            'мы': 'организуем', 'вы': 'организуете', 'они': 'организуют'
        },
        'past': {
            'мужской род': 'организовал', 'женский род': 'организовала',
            'средний род': 'организовало', 'множественное число': 'организовали'
        }
    },

    'исследовать': {
        'present': {
            'я': 'исследую', 'ты': 'исследуешь', 'он/она': 'исследует',
            'мы': 'исследуем', 'вы': 'исследуете', 'они': 'исследуют'
        },
        'past': {
            'мужской род': 'исследовал', 'женский род': 'исследовала',
            'средний род': 'исследовало', 'множественное число': 'исследовали'
        }
    },

    'замолчать': {
        'future_simple': {
            'я': 'замолчу', 'ты': 'замолчишь', 'он/она': 'замолчит',
            'мы': 'замолчим', 'вы': 'замолчите', 'они': 'замолчат'
        },
        'past': {
            'мужской род': 'замолчал', 'женский род': 'замолчала',
            'средний род': 'замолчало', 'множественное число': 'замолчали'
        }
    },

    'мыться': {
        'present': {
            'я': 'моюсь', 'ты': 'моешься', 'он/она': 'моется',
            'мы': 'моемся', 'вы': 'моетесь', 'они': 'моются'
        },
        'past': {
            'мужской род': 'мылся', 'женский род': 'мылась',
            'средний род': 'мылось', 'множественное число': 'мылись'
        }
    },
}

class SimpleFST:

    def __init__(self):

        self.lexicon: Dict[str, Tuple[str, str, str, Dict[str, Dict]]] = {}
        self._verbs_loaded = False  # предотвращает повторные загрузки
        # вспомогательный глагол для сложного будущего
        self.compound_aux = {
            '1sg': 'буду', '2sg': 'будешь', '3sg': 'будет',
            '1pl': 'будем', '2pl': 'будете', '3pl': 'будут'
        }

        self.endings = {
            '1': {
                'present': pynini.string_map([
                    ("1sg", "ю"), ("2sg", "ешь"), ("3sg", "ет"),
                    ("1pl", "ем"), ("2pl", "ете"), ("3pl", "ют")
                ]),
                'past': pynini.string_map([
                    ("masc", "л"), ("femn", "ла"), ("neut", "ло"), ("plur", "ли")
                ])
            },
            '2': {
                'present': pynini.string_map([
                    ("1sg", "ю"), ("2sg", "ишь"), ("3sg", "ит"),
                    ("1pl", "им"), ("2pl", "ите"), ("3pl", "ят")
                ]),
                'past': pynini.string_map([
                    ("masc", "л"), ("femn", "ла"), ("neut", "ло"), ("plur", "ли")
                ])
            }
        }

    # --- служебные ---
    @staticmethod
    def _compute_past_stem(inf: str) -> str:
        inf = inf.strip().lower()
        if inf.endswith('ться'):
            return inf[:-4]
        if inf.endswith('ть'):
            return inf[:-2]
        return inf

    @staticmethod
    def _detect_conj(inf: str) -> str:
        return '2' if inf.endswith('ить') else '1'

    def _lazy_load_verbs(self):

        if self._verbs_loaded:
            return
        try:
            from builtins import fallback_verbs
            for verb in fallback_verbs:
                self.add_verb(verb)
            self._verbs_loaded = True
        except Exception:
            return

    def load_verbs(self, verbs_list):

        if not verbs_list:
            return
        for v in verbs_list:
            self.add_verb(v)
        self._verbs_loaded = True

        self._lazy_load_verbs()

    def add_verb(self, infinitive: str, conj_class: Optional[str] = None,
             aspect: str = 'impf', explicit: Optional[Dict[str, Dict]] = None):

        if not infinitive:
            return

        infinitive = infinitive.strip().lower()

        # --- проверяем, возвратный ли глагол ---
        is_reflexive = infinitive.endswith(('ся', 'сь'))
        base_inf = infinitive[:-2] if is_reflexive else infinitive

        # --- определяем спряжение и основу ---
        conj = conj_class or self._detect_conj(base_inf)
        past_stem = self._compute_past_stem(base_inf)

        # --- определяем вид ---
        if base_inf in PERFECTIVE_VERBS:
            aspect = 'perf'
        elif base_inf in IMPERFECTIVE_VERBS:
            aspect = 'impf'
        elif base_inf in BIA_VERBS:
            aspect = 'biaspectual'
        else:

            if base_inf.startswith(("по", "с", "за", "при", "от", "вы", "вз", "раз", "у")):
                aspect = 'perf'
            elif base_inf.endswith(("ировать", "овать")):
                aspect = 'biaspectual'
            else:
                aspect = 'impf'

        # --- определяем основу настоящего времени ---
        if base_inf in STEM_EXCEPTIONS:
            present_stem = STEM_EXCEPTIONS[base_inf]
        else:
            present_stem = past_stem
            for k, v in ALTERNATIONS.items():
                if past_stem.endswith(k):
                    present_stem = past_stem[:-len(k)] + v
                    break

        # --- объединяем явные формы ---
        explicit_defaults = EXPLICIT_VERB_FORMS.get(base_inf, {}) or {}
        merged_explicit = dict(explicit_defaults)
        if explicit and isinstance(explicit, dict):
            merged_explicit.update(explicit)

        # --- сохраняем в лексикон ---
        self.lexicon[base_inf] = (past_stem, conj, aspect, merged_explicit)


        if is_reflexive:
            self.lexicon[infinitive] = (past_stem, conj, aspect, merged_explicit)


    # --- построение FST ---
    def _build_fst(self, stem: str, conj: str, tense: str) -> pynini.Fst:
        endings_fst = self.endings[conj][tense]
        return pynini.cross("", stem) + endings_fst

    # --- синтез ---
    def synthesize(self, infinitive: str) -> Dict[str, Dict]:
        self._lazy_load_verbs()
        start = time.time()
        infinitive = (infinitive or "").strip().lower()
        if not infinitive or infinitive not in self.lexicon:
            return {}


        orig_inf = infinitive

        base_inf = orig_inf[:-2] if orig_inf.endswith(('ся', 'сь')) else orig_inf

        past_stem, conj, aspect, explicit = self.lexicon[infinitive]


        if base_inf in STEM_EXCEPTIONS:
            present_stem = STEM_EXCEPTIONS[base_inf]
        else:
            present_stem = past_stem
            for k, v in ALTERNATIONS.items():
                if past_stem.endswith(k):
                    present_stem = past_stem[:-len(k)] + v
                    break

        result = {
            'infinitive': {'форма': orig_inf,
                           'вид': 'совершенный' if aspect == 'perf'
                                  else 'двувидовой' if aspect == 'biaspectual'
                                  else 'несовершенный'},
            'past': {}, 'present': {}, 'future_simple': {}, 'future_compound': {}
        }

        # --- прошедшее ---
        past_fst = self._build_fst(past_stem, conj, 'past')
        for tag in ['masc', 'femn', 'neut', 'plur']:
            try:
                word = rewrite.one_top_rewrite(tag, past_fst)
                label = {'masc': 'мужской род', 'femn': 'женский род',
                         'neut': 'средний род', 'plur': 'множественное число'}[tag]
                result['past'][label] = word
            except pynini.FstOpError:
                continue

        # --- настоящее / простое будущее ---
        present_fst = self._build_fst(present_stem, conj, 'present')
        person_map = {'1sg': 'я', '2sg': 'ты', '3sg': 'он/она',
                      '1pl': 'мы', '2pl': 'вы', '3pl': 'они'}

        def adjust_phonetics(stem: str, form: str) -> str:
            """Исправляет неправильные окончания по фонетическим правилам."""
            if not form:
                return form

            if any(stem.endswith(c) for c in "жшщч"):
                form = form.replace("я", "а").replace("ю", "у")

            if any(stem.endswith(c) for c in "жшщч") and form.endswith("ют"):
                form = form[:-2] + "ут"

            if stem.endswith(("д", "в")):
                if form.endswith("ю"):
                    form = form[:-1] + "у"
                elif form.endswith("ют"):
                    form = form[:-2] + "ут"
            return form

        # вспомогательная функция для корректного присоединения возвратной частицы
        def attach_reflexive(word: str, block: str,person_key: str) -> str:

            if not word:
                return word
            if word.endswith(('ся', 'сь')):
                return word

            if block in ('present', 'future_simple'):
                if person_key  in ('я', 'вы'):
                    return word + 'сь'
                else:
                    return word + 'ся'
            elif block == 'past':
                if person_key == 'мужской род':
                    return word + 'ся'
                else:
                    return word + 'сь'
            return word

        if aspect == 'impf':
            for tag in person_map:
                try:
                    word = rewrite.one_top_rewrite(tag, present_fst)
                except pynini.FstOpError:
                    word = ""
                word = adjust_phonetics(present_stem, word)
                if word:
                    result['present'][person_map[tag]] = word
        elif aspect == 'perf':
            for tag in person_map:
                try:
                    word = rewrite.one_top_rewrite(tag, present_fst)
                except pynini.FstOpError:
                    word = ""
                word = adjust_phonetics(present_stem, word)
                if word:
                    result['future_simple'][person_map[tag]] = word
        else:
            for tag in person_map:
                try:
                    word = rewrite.one_top_rewrite(tag, present_fst)
                except pynini.FstOpError:
                    word = ""
                word = adjust_phonetics(present_stem, word)
                if word:
                    result['present'][person_map[tag]] = word

        # --- сложное будущее ---
        if aspect in ('impf', 'biaspectual'):
            for tag, aux in self.compound_aux.items():

                result['future_compound'][person_map[tag]] = f"{aux} {orig_inf}"


        if explicit:
            for tense_key, forms in explicit.items():
                if tense_key in result and isinstance(forms, dict):
                    result[tense_key].update(forms)

        # Добавление рефлексивной частицы

        if orig_inf.endswith(('ся', 'сь')):
            for block in ('present', 'future_simple', 'past'):
                for person_key in list(result.get(block, {}).keys()):
                    val = result[block].get(person_key)
                    if not val:
                        continue

                    if val.endswith('ся') or val.endswith('сь'):
                        continue
                    result[block][person_key] = attach_reflexive(val, block, person_key)

        # метаданные
        result['_meta'] = {'model': 'simple-fst', 'elapsed_seconds': time.time() - start}
        result['source'] = 'simple-fst'
        return result

    # --- анализ ---
    def analyze(self, form: str):
        """Генерация всех форм и поиск совпадения."""
        self._lazy_load_verbs()
        start = time.time()
        form = (form or "").strip().lower()
        if not form:
            return None

        person_label_map = {
            'я': '1 лицо, ед. число', 'ты': '2 лицо, ед. число', 'он/она': '3 лицо, ед. число',
            'мы': '1 лицо, мн. число', 'вы': '2 лицо, мн. число', 'они': '3 лицо, мн. число'
        }
        pronoun_to_tag = {'я': '1sg', 'ты': '2sg', 'он/она': '3sg', 'мы': '1pl', 'вы': '2pl', 'они': '3pl'}
        past_label_to_gender = {
            'мужской род': ('masc', 'sing'), 'женский род': ('femn', 'sing'),
            'средний род': ('neut', 'sing'), 'множественное число': (None, 'plur')
        }

        # --- сложное будущее ---
        prefixes = {'буду': '1sg', 'будешь': '2sg', 'будет': '3sg',
                    'будем': '1pl', 'будете': '2pl', 'будут': '3pl'}
        for pref, tag in prefixes.items():
            if form.startswith(pref + ' '):
                for lemma, (past_stem, conj, aspect_flag, explicit) in self.lexicon.items():
                    synth = self.synthesize(lemma)
                    cmp = synth.get('future_compound', {})
                    if cmp and any(v == form for v in cmp.values()):
                        aspect_human = ('совершенный' if aspect_flag == 'perf'
                                        else 'двувидовой' if aspect_flag == 'biaspectual'
                                        else 'несовершенный')
                        res = {
                            "инфинитив": lemma,
                            "время": "будущее (сложное)",
                            "аспект": aspect_human,
                            "aspect": aspect_flag,
                            "person": tag,
                            "лицо/число": person_label_map.get(tag),
                            "gender": None,
                            "number": 'sing' if tag.endswith('sg') else 'plur',
                            "форма": form
                        }
                        res['_meta'] = {'model': 'simple-fst', 'elapsed_seconds': time.time() - start}
                        return res

        # --- общий перебор ---
        for lemma in self.lexicon:
            synth = self.synthesize(lemma)
            for k, w in synth.get('past', {}).items():
                if w == form:
                    aspect_human = synth['infinitive'].get('вид', 'несовершенный')
                    if 'двув' in aspect_human:
                        aspect_flag = 'biaspectual'
                    elif 'соверш' in aspect_human:
                        aspect_flag = 'perf'
                    else:
                        aspect_flag = 'impf'
                    gender, number = past_label_to_gender.get(k, (None, None))
                    res = {
                        "инфинитив": lemma,
                        "время": "прошедшее",
                        "аспект": aspect_human,
                        "aspect": aspect_flag,
                        "род/число": k,
                        "gender": gender,
                        "number": number,
                        "person": None,
                        "форма": form
                    }
                    res['_meta'] = {'model': 'simple-fst', 'elapsed_seconds': time.time() - start}
                    return res

            for block in ('present', 'future_simple'):
                for k, w in synth.get(block, {}).items():
                    if w == form:
                        tense = "настоящее" if block == 'present' else "простое будущее"
                        aspect_human = synth['infinitive'].get('вид', 'несовершенный')
                        if 'двув' in aspect_human:
                            aspect_flag = 'biaspectual'
                        elif 'соверш' in aspect_human:
                            aspect_flag = 'perf'
                        else:
                            aspect_flag = 'impf'
                        person_tag = pronoun_to_tag.get(k)
                        res = {
                            "инфинитив": lemma,
                            "время": tense,
                            "аспект": aspect_human,
                            "aspect": aspect_flag,
                            "person": person_tag,
                            "лицо/число": person_label_map.get(k),
                            "gender": None,
                            "number": 'sing' if person_tag and person_tag.endswith('sg') else 'plur',
                            "форма": form
                        }
                        res['_meta'] = {'model': 'simple-fst', 'elapsed_seconds': time.time() - start}
                        return res

            for k, w in synth.get('future_compound', {}).items():
                if w == form:
                    aspect_human = synth['infinitive'].get('вид', 'несовершенный')
                    if 'двув' in aspect_human:
                        aspect_flag = 'biaspectual'
                    elif 'соверш' in aspect_human:
                        aspect_flag = 'perf'
                    else:
                        aspect_flag = 'impf'
                    person_tag = pronoun_to_tag.get(k)
                    res = {
                        "инфинитив": lemma,
                        "время": "будущее (сложное)",
                        "аспект": aspect_human,
                        "aspect": aspect_flag,
                        "person": person_tag,
                        "лицо/число": person_label_map.get(k),
                        "gender": None,
                        "number": 'sing' if person_tag and person_tag.endswith('sg') else 'plur',
                        "форма": form
                    }
                    res['_meta'] = {'model': 'simple-fst', 'elapsed_seconds': time.time() - start}
                    return res

        return None


In [ ]:
# Ячейка 3: FSTProcessor
import pynini
from pynini.lib import rewrite
from typing import Optional
import time


class FSTProcessor:

    def __init__(self):
        self.fst = SimpleFST()
        # небольшой initial seed-лексикон — чтобы интерфейс работал сразу
        seeds = [
            ('читать', '1', 'impf', {}),
            ('писать', '1', 'impf', {'пишу': {'tag': 'PRES_1SG'}}),
            ('прочитать', '1', 'perf', {}),
            ('купить', '2', 'perf', {'купил': {'tag': 'PAST_MASC'}, 'купила': {'tag': 'PAST_FEM'}, 'купили': {'tag': 'PAST_PL'}}),
            ('покупать', '1', 'impf', {}),
            ('говорить', '2', 'impf', {}),
            ('сказать', '2', 'perf', {}),
            ('идти', '1', 'impf', {'шел': {'tag': 'PAST_MASC'}, 'шла': {'tag': 'PAST_FEM'}, 'шли': {'tag': 'PAST_PL'}}),
            ('ходить', '1', 'impf', {}),
            ('спать', '1', 'impf', {'сплю': {'tag': 'PRES_1SG'}, 'спал': {'tag': 'PAST_MASC'}, 'спали': {'tag': 'PAST_PL'}}),
        ]
        # Попытка загрузить частый список глаголов:
        try:

            fbv = globals().get("fallback_verbs", None)
            if not fbv and "get_all_verbs" in globals():
                try:
                    fbv = get_all_verbs()
                except Exception:
                    fbv = None

            if fbv:

                if hasattr(self.fst, "load_verbs"):
                    self.fst.load_verbs(fbv)
                else:
                    for v in fbv:
                        try:
                            self.fst.add_verb(v)
                        except Exception:
                            continue
        except Exception as e:
            print(f" Не удалось загрузить fallback_verbs автоматически: {e}")

        # обучение
        self._fst_trained = False
        self._trained_synth_fst = None
        self._trained_analyze_fst = None
        self._trained_pairs = []

    # --- Вспомогательные: маппинг меток (label -> tag) ---
    @staticmethod
    def map_label_to_tag(label: str, block: str) -> str:
        lbl = (label or "").lower()
        if block == "past":
            if "муж" in lbl: return "PST+MASC"
            if "жен" in lbl: return "PST+FEM"
            if "сред" in lbl: return "PST+NEUT"
            if "мн" in lbl: return "PST+PL"
            return "PST"
        elif block in ("present", "future_simple"):
            if "я" in lbl: return "PRS+1SG"
            if "ты" in lbl: return "PRS+2SG"
            if "он" in lbl or "она" in lbl or "он/она" in lbl: return "PRS+3SG"
            if "мы" in lbl: return "PRS+1PL"
            if "вы" in lbl: return "PRS+2PL"
            if "они" in lbl: return "PRS+3PL"
            return "PRS"
        elif block == "future_compound":
            if "я" in lbl: return "FUT+CMP+1SG"
            if "ты" in lbl: return "FUT+CMP+2SG"
            if "он" in lbl or "она" in lbl or "он/она" in lbl: return "FUT+CMP+3SG"
            if "мы" in lbl: return "FUT+CMP+1PL"
            if "вы" in lbl: return "FUT+CMP+2PL"
            if "они" in lbl: return "FUT+CMP+3PL"
            return "FUT+CMP"
        return "UNK"

    # --- вспомогательное: парсинг тега FST
    @staticmethod
    def parse_tag(tagged: str) -> Optional[VerbForm]:
        if not tagged or '+' not in tagged:
            return None

        parts = tagged.split('+')
        infinitive = parts[0]
        tags = parts[1:]
        tense, aspect, person, gender, number = None, None, None, None, None

        for t in tags:
            t = t.upper()

            # --- ВИД ---
            if t == 'PERF':
                aspect = 'perf'
            elif t == 'IMPF':
                aspect = 'impf'
            elif t == 'BIA':
                aspect = 'biaspectual'

            # --- ВРЕМЯ ---
            if t.startswith('PST'):
                tense = 'past'
            elif t.startswith('PRS'):
                tense = 'present'
            elif t.startswith('FUT'):
                tense = 'future'

            # --- ЛИЦО / ЧИСЛО ---
            if t.endswith('1SG'):
                person, number = '1sg', 'sing'
            elif t.endswith('2SG'):
                person, number = '2sg', 'sing'
            elif t.endswith('3SG'):
                person, number = '3sg', 'sing'
            elif t.endswith('1PL'):
                person, number = '1pl', 'plur'
            elif t.endswith('2PL'):
                person, number = '2pl', 'plur'
            elif t.endswith('3PL'):
                person, number = '3pl', 'plur'

            # --- РОД / ЧИСЛО ---
            elif t.endswith('MASC'):
                gender, number = 'masc', 'sing'
            elif t.endswith('FEM'):
                gender, number = 'femn', 'sing'
            elif t.endswith('NEUT'):
                gender, number = 'neut', 'sing'
            elif t.endswith('PL'):
                number = 'plur'

        # Значения по умолчанию
        if aspect is None:
            aspect = 'impf'
        if tense is None:
            tense = 'present'

        return VerbForm(
            infinitive=infinitive,
            tense=tense,
            aspect=aspect,
            person=person,
            gender=gender,
            number=number,
            form=""
        )
    def ensure_3pl_from_3sg(self):
        """
        Постпроцессинг: если в обученных парах нет формы 3PL (они),
        но есть форма 1SG (я), добавляем 3PL = 1SG + 'т'.
        """
        new_pairs = []
        seen = {k for k, _ in self._trained_pairs}

        # группируем по инфинитиву
        grouped = {}
        for k, v in self._trained_pairs:
            lemma = k.split('+')[0]
            grouped.setdefault(lemma, []).append((k, v))

        for lemma, forms in grouped.items():
            for k, v in forms:
                if '+PRS+1SG' in k or '+PERF+PRS+1SG' in k or '+IMPF+PRS+1SG' in k:

                    k3pl = k.replace('+1SG', '+3PL')
                    if k3pl not in seen:
                        f3pl = v if v.endswith('т') else v + 'т'
                        new_pairs.append((k3pl, f3pl))
                        seen.add(k3pl)
        if new_pairs:
            print(f" Добавлено {len(new_pairs)} новых форм 3PL (они = 1SG + 'т')")
            self._trained_pairs.extend(new_pairs)


    # --- Обучение:
    def train_from_pymorphy_processor(self, infinitives=None, max_verbs=None):

        try:
            pym = get_pymorphy_processor()
        except Exception as e:
            raise RuntimeError(f"Невозможно обучить FST: pymorphy3 недоступен ({e})")

        if infinitives is None:
            raise ValueError("Передайте список инфинитивов в аргумент infinitives.")

        start = time.time()

        pairs, seen = [], set()
        count = 0
        for inf in infinitives:
            if max_verbs and count >= max_verbs:
                break
            count += 1
            inf = inf.strip().lower()

            try:
                synth = pym.synthesize(inf)
            except Exception:
                synth = {}

            if not synth:
                key = f"{inf}+IMPF+INFN"
                if key not in seen:
                    pairs.append((key, inf))
                    seen.add(key)
                continue

            # --- определяем вид глагола ---
            asp = synth.get('infinitive', {}).get('вид', '').lower()
            if 'двувид' in asp:
                aspect_tag = 'BIA'
            elif 'соверш' in asp and not asp.startswith('не'):
                aspect_tag = 'PERF'
            else:
                aspect_tag = 'IMPF'

            # --- инфинитив ---
            key_inf = f"{inf}+{aspect_tag}+INFN"
            if key_inf not in seen:
                pairs.append((key_inf, synth['infinitive'].get('форма', inf)))
                seen.add(key_inf)

            # --- блоки времени ---
            for block in ("past", "present", "future_simple", "future_compound"):
                sub = synth.get(block, {})
                if isinstance(sub, dict):
                    for label, form in sub.items():
                        if not form:
                            continue
                        tag = self.map_label_to_tag(label, block)
                        key = f"{inf}+{aspect_tag}+{tag}"
                        if key not in seen:
                            pairs.append((key, form))
                            seen.add(key)

                    # --- добавляем 3PL (они), если её нет ---
                    if 'они' not in sub:
                        # попробуем вывести её из других форм
                        if 'мы' in sub:
                            form = sub['мы']
                        elif 'вы' in sub:
                            form = sub['вы']
                        elif 'множественное число' in sub:
                            form = sub['множественное число']
                        else:
                            form = None
                        if form:
                            if block == 'future_compound':
                                tag = 'FUT+CMP+3PL'
                            elif block == 'future_simple':
                                tag = 'FUT+3PL'
                            elif block == 'present':
                                tag = 'PRS+3PL'
                            elif block == 'past':
                                tag = 'PST+PL'
                            else:
                                tag = None
                            if tag:
                                key = f"{inf}+{aspect_tag}+{tag}"
                                if key not in seen:
                                    pairs.append((key, form))
                                    seen.add(key)

        if not pairs:
            raise RuntimeError("Не удалось собрать пары для обучения FST")

        self._trained_pairs = pairs[:]
        try:
            self._trained_synth_fst = pynini.string_map(sorted(pairs)).optimize()
            self._trained_analyze_fst = pynini.invert(self._trained_synth_fst).optimize()
            self._fst_trained = True
            elapsed = time.time() - start
            print(f" FST обучен на {len(pairs)} парах (время: {elapsed:.2f}s)")
            return {"pairs": len(pairs), "_meta": {"elapsed_seconds": elapsed}}
        except Exception as e:
            raise RuntimeError(f"Ошибка при построении FST: {e}")



    # --- Анализ ---
    def analyze(self, form: str) -> Optional[VerbForm]:
        start = time.time()
        form = (form or "").strip().lower()
        if not form:
            return None

        # 1) обученный FST
        if getattr(self, "_fst_trained", False) and self._trained_analyze_fst is not None:
            try:
                rewrites = rewrite.top_rewrites(form, self._trained_analyze_fst, 5)
                if not rewrites:
                    direct_hits = [k for k, v in self._trained_pairs if v == form]
                    if direct_hits:
                        rewrites = direct_hits

            except Exception as e:

                rewrites = []
            if rewrites:
                vf = self.parse_tag(rewrites[0])
                if vf:
                    vf.form = form
                    vf.source = "trained-fst"
                    vf._meta = {"model": "fst+pymorphy3", "elapsed_seconds": time.time() - start}  # ADDED META
                    return vf

        # 2) pymorphy3 fallback
        try:
            pym = get_pymorphy_processor()
            res = pym.analyze(form)
            if res:
                res.source = "pymorphy3"
                res._meta = {"model": "pymorphy3", "elapsed_seconds": time.time() - start}  # ADDED META
                return res
        except Exception:
            pass

        # 3) локальный SimpleFST.analyze
        res = self.fst.analyze(form)
        if not res:
            return None


        infinitive = res.get("инфинитив") or res.get("infinitive") or res.get("lemma")
        aspect_raw = res.get("аспект") or res.get("aspect") or 'impf'
        if aspect_raw == 'двувидовой' or aspect_raw == 'biaspectual':
            aspect = 'biaspectual'
        elif aspect_raw == 'совершенный' or aspect_raw == 'perf':
            aspect = 'perf'
        else:
            aspect = 'impf'

        time_str = (res.get("время","") or res.get("time","")).lower()
        if "прошед" in time_str or "past" in time_str:
            tense = 'past'
        elif "будущее" in time_str or "future" in time_str:
            tense = 'future'
        else:
            tense = 'present'

        person = None
        gender = None
        number = None

        lp = res.get("лицо/число") or res.get("лицо") or res.get("person")
        if lp:
            person = lp
            number = 'sing' if lp.endswith('sg') else 'plur'

        rn = res.get("род/число") or res.get("род") or res.get("gender")
        if rn:
            if rn in ('masc','femn','neut'):
                gender = rn
                number = 'sing'
            elif rn == 'plur':
                number = 'plur'

        form_str = res.get("форма") or res.get("form") or form

        vf = VerbForm(
            infinitive=infinitive,
            tense=tense,
            aspect=aspect,
            person=person,
            gender=gender,
            number=number,
            form=form_str
        )

        vf.source = "fst"
        vf._meta = {"model": "fst+pymorphy3", "elapsed_seconds": time.time() - start}
        return vf

    def fst_synthesize_inf(self, infinitive: str) -> Dict[str, Dict]:

        infinitive = (infinitive or "").strip().lower()
        if not getattr(self, "_fst_trained", False) or not hasattr(self, "_trained_pairs"):
            return {}

        # Собираем все пары, относящиеся к данному инфинитиву
        entries = [(k, v) for k, v in self._trained_pairs if k.startswith(infinitive + "+")]
        if not entries:
            return {}

        # Базовая структура
        result = {
            'infinitive': {'форма': infinitive, 'вид': 'несовершенный'},
            'past': {}, 'present': {}, 'future_simple': {}, 'future_compound': {}
        }

        # Карты для лиц/родов
        person_map = {
            '1SG': 'я', '2SG': 'ты', '3SG': 'он/она',
            '1PL': 'мы', '2PL': 'вы', '3PL': 'они'
        }
        gender_map = {
            'MASC': 'мужской род', 'FEM': 'женский род',
            'NEUT': 'средний род', 'PL': 'множественное число'
        }

        # --- 1) Попытка определить аспект прямо из ключей ---
        aspect_counts = {'PERF': 0, 'IMPF': 0, 'BIA': 0}
        for k, _ in entries:
            parts = [p.upper() for p in k.split('+') if p]
            if len(parts) >= 2 and parts[1] in aspect_counts:
                aspect_counts[parts[1]] += 1

        explicit_aspect = None

        for a in ('PERF', 'IMPF', 'BIA'):
            if aspect_counts[a] > 0:
                explicit_aspect = a
                break

        if explicit_aspect == 'PERF':
            result['infinitive']['вид'] = 'совершенный'
        elif explicit_aspect == 'IMPF':
            result['infinitive']['вид'] = 'несовершенный'
        elif explicit_aspect == 'BIA':
            result['infinitive']['вид'] = 'двувидовой'

        # --- 2) Основной проход по парам:  ---
        for k, form in entries:
            tagpart = k[len(infinitive) + 1:]
            parts = [p.upper() for p in tagpart.split('+') if p]
            if not parts:
                continue


            aspect_tok = None
            tokens = parts[:]
            if parts[0] in ('PERF', 'IMPF', 'BIA'):
                aspect_tok = parts[0]
                tokens = parts[1:]

            # Определяем блок времени
            block = None
            up_tokens = tokens
            if any('FUT+CMP' in tok for tok in up_tokens):
                block = 'future_compound'
            elif any(tok.startswith('PST') for tok in up_tokens):
                block = 'past'
            elif any(tok.startswith('PRS') for tok in up_tokens):

                if aspect_tok == 'PERF' or explicit_aspect == 'PERF':
                    block = 'future_simple'
                else:
                    block = 'present'
            elif any(tok.startswith('FUT') for tok in up_tokens):
                block = 'future_simple'

            if not block:
                continue


            if block == 'past':

                placed = False
                for tok in up_tokens:
                    for g in gender_map:
                        if g in tok:
                            result['past'][gender_map[g]] = form
                            placed = True
                            break
                    if placed:
                        break
                if not placed:

                    result['past']['множественное число'] = form
                continue

            if block in ('present', 'future_simple'):
                placed = False
                for tok in up_tokens:
                    for p in person_map:
                        if p in tok:
                            target = person_map[p]
                            result[block][target] = form
                            placed = True
                            break
                    if placed:
                        break
                continue

            if block == 'future_compound':
                placed = False
                for tok in up_tokens:
                    for p in person_map:
                        if p in tok:
                            target = person_map[p]
                            result['future_compound'][target] = form
                            placed = True
                            break
                    if placed:
                        break
                continue

        # --- 3) Дополнительный проход: если для какого-то блока 'они' отсутствует,
        # попробовать найти любые пары с 3PL и использовать их (не затирая существующие) ---
        for k, form in entries:
            ku = k.upper()
            if '3PL' not in ku:
                continue
            # определим тип блока по содержимому ключа
            # варианты: PST+PL, PRS+3PL, FUT+CMP+3PL, FUT+3PL (и т.п.)
            if 'PST' in ku and 'PL' in ku:
                # past множественное — это уже handled via past 'множественное число', но на всякий случай
                if 'множественное число' not in result['past']:
                    result['past']['множественное число'] = form
            elif 'FUT+CMP' in ku:
                if 'они' not in result['future_compound']:
                    result['future_compound']['они'] = form
            elif 'PRS' in ku:
                if 'они' not in result['present']:
                    result['present']['они'] = form
            elif 'FUT' in ku:
                # FUT (простое)
                if 'они' not in result['future_simple']:
                    result['future_simple']['они'] = form
            else:
                # на всякий случай: если ключ содержит просто '+3PL' (непонятно где), пробуем вставить в future_compound/present
                if 'они' not in result['present']:
                    result['present']['они'] = form

        # --- 4) Если аспект всё ещё не установлен явно — используем консервативную эвристику ---
        if explicit_aspect is None:
            has_present = bool(result['present'])
            has_future_simple = bool(result['future_simple'])
            has_future_compound = bool(result['future_compound'])
            if has_present and has_future_compound:
                result['infinitive']['вид'] = 'двувидовой'
            elif has_future_simple and not has_present:
                result['infinitive']['вид'] = 'совершенный'
            else:
                result['infinitive']['вид'] = 'несовершенный'

        return result



    def synthesize(self, infinitive: str) -> Dict[str, Dict]:

        start = time.time()
        infinitive = (infinitive or "").strip().lower()
        # 1) если обученный FST доступен
        if getattr(self, "_fst_trained", False):
            res = self.fst_synthesize_inf(infinitive)
            if res:
                res["source"] = "trained-fst"
                res["_meta"] = {"model": "fst+pymorphy3", "elapsed_seconds": time.time() - start}  # ADDED META
                return res

        # 2) pymorphy3
        try:
            pym = get_pymorphy_processor()
            res = pym.synthesize(infinitive)
            if res:
                res["source"] = "pymorphy3"
                res["_meta"] = {"model": "pymorphy3", "elapsed_seconds": time.time() - start}  # ADDED META
                return res

        except Exception:
            pass

        # 3) локальный SimpleFST
        res = self.fst.synthesize(infinitive)
        if res:
            res["source"] = "fst"
            res["_meta"] = {"model": "fst", "elapsed_seconds": time.time() - start}  # ADDED META
        return res



_pym_processor = None
_fst_processor = None

def get_pymorphy_processor():
    global _pym_processor
    if _pym_processor is None:
        if not pymorphy3:
            raise RuntimeError("pymorphy3 недоступен в окружении.")
        _pym_processor = VerbProcessor()
    return _pym_processor

def get_fst_processor():
    global _fst_processor
    if _fst_processor is None:
        _fst_processor = FSTProcessor()
    return _fst_processor


In [ ]:
# Ячейка 4: утилиты обучения/загрузки и CLI
import os
import pynini
import time


def get_all_verbs(limit=None):

    verbs = set()

    if pymorphy3 is not None:
        try:
            morph = pymorphy3.MorphAnalyzer()

            if hasattr(morph.dictionary, "iter_known_words"):
                for lemma in morph.dictionary.iter_known_words():
                    for p in morph.parse(lemma):
                        if 'INFN' in p.tag or 'VERB' in p.tag:
                            verbs.add(p.normal_form)
            else:

                raise AttributeError("iter_known_words отсутствует")
        except Exception:
            verbs = set()
    if not verbs:
        fallback_verbs = [
            # --- базовые ---
            "быть", "иметь", "делать", "сказать", "говорить", "видеть", "идти", "стоять", "сидеть", "лежать",
            "думать", "знать", "понимать", "работать", "жить", "ехать", "пить", "есть", "писать", "читать",
            "любить", "ненавидеть", "спать", "играть", "взять", "дать", "купить", "покупать", "продавать",
            "учиться", "учить", "помнить", "забыть", "ждать", "ходить", "приходить", "уходить", "бежать",
            "смотреть", "слышать", "написать", "прочитать", "встать", "лечь", "начать", "закончить", "выйти",
            "войти", "вздохнуть", "улыбнуться", "плакать", "смеяться", "уметь", "мочь", "решить", "проверить",
            "искать", "найти", "встретить", "спросить", "ответить", "объяснить", "понять",
            "хотеть", "пойти", "прийти", "принести", "нести", "посмотреть", "послушать",
            "танцевать", "петь", "рисовать", "готовить", "открыть", "закрыть", "позвонить",
            "достать", "класть", "держать", "вести", "плыть", "лететь", "летать", "дудеть",

            # --- возвратные ---
            "учиться", "смеяться", "улыбаться", "бояться", "надеяться", "казаться", "смотреться",
            "пользоваться", "заниматься", "приближаться", "отдаляться", "встречаться", "общаться",
            "прятаться", "сдаваться", "двигаться", "перемещаться", "находиться", "прощаться",
            "развиваться", "мыться", "бриться", "одеваться", "раздеваться", "кушаться", "собираться",
            "жениться", "увольняться", "здороваться", "извиняться", "радоваться", "удивляться",
            "обниматься", "целоваться", "знакомиться", "переписываться", "ссориться", "мириться",
            "интересоваться", "называться", "остановиться", "двигаться", "продаваться", "бояться",

            # --- двувидовые ---
            "организовать", "атаковать", "исследовать",

            # --- частотные пары (сов/несов) ---
            "строить", "построить", "ломать", "сломать", "брать", "взять", "давать", "дать",
            "кричать", "закричать", "молчать", "замолчать", "помогать", "помочь", "мешать", "помешать",
            "читать", "прочитать", "писать", "написать", "говорить", "сказать", "смотреть", "посмотреть",
            "готовить", "приготовить", "начинать", "начать", "заканчивать", "закончить",
            "решать", "решить", "забывать", "забыть", "проверять", "проверить", "звонить", "позвонить",
            "смотреть", "увидеть", "слышать", "услышать", "ждать", "подождать", "бежать", "прибежать",
            "приходить", "прийти", "уходить", "уйти", "приносить", "принести", "класть", "положить",

            # --- редкие, но полезные ---
            "сохранять", "передавать", "получать", "выбирать", "поддерживать", "обслуживать", "победить",
            "доставлять", "замечать", "повторять", "соглашаться", "отказываться", "падать", "упасть",
            "держаться", "касаться", "стараться", "находить", "терять", "менять", "заменять", "узнавать",
            "отдыхать", "гулять", "бродить", "кататься", "вставать", "просыпаться", "засыпать"
        ]

        verbs = set(fallback_verbs)

    res = sorted(list(verbs))
    if limit:
        res = res[:limit]
    return res


def train_and_save_fst(fst_proc, limit=5000):

    verbs = get_all_verbs(limit=limit)
    print(f" Найдено {len(verbs)} глаголов (limit={limit}), начинаем обучение FST...")
    fst_proc.train_from_pymorphy_processor(infinitives=verbs, max_verbs=limit)

    # Сохраняем автоматы
    try:
        fst_proc._trained_synth_fst.write("trained_synth.fst")
        fst_proc._trained_analyze_fst.write("trained_analyze.fst")
        print(" FST сохранён в файлы trained_synth.fst и trained_analyze.fst.")
    except Exception as e:
        print(" Ошибка при сохранении FST:", e)

def load_trained_fst(fst_proc: FSTProcessor, prefix="trained"):

    try:
        synth_path = f"{prefix}_synth.fst"
        analyze_path = f"{prefix}_analyze.fst"
        if os.path.exists(synth_path) and os.path.exists(analyze_path):
            fst_proc._trained_synth_fst = pynini.Fst.read(synth_path)
            fst_proc._trained_analyze_fst = pynini.Fst.read(analyze_path)
            fst_proc._fst_trained = True
            print(" Trained FSTs loaded.")
            return True
    except Exception as e:
        print("Ошибка загрузки trained FST:", e)
    return False


def pretty_print_synth(forms: Dict[str, Dict], infinitive: str):
    if not forms:
        print(f" Глагол '{infinitive}' не найден или не является глаголом.")
        return

    inf_info = forms.get('infinitive', {})
    vid = inf_info.get('вид', 'несовершенный')
    print(f"\n Формы глагола '{infinitive}':\n")
    print(f" Инфинитив: {inf_info.get('форма', infinitive)} ({vid} вид)\n")

    if forms.get('past'):
        print(" Прошедшее время:")

        order = ['мужской род', 'женский род', 'средний род', 'множественное число']
        for label in order:
            if label in forms['past']:
                print(f"   {label}: {forms['past'][label]}")
        print()


    if vid == 'двувидовой':
        merged = {}

        if forms.get('present'):
            merged = forms['present']
        elif forms.get('future_simple'):
            merged = forms['future_simple']
        if merged:
            print(" Настоящее/Простое будущее:")
            person_order = ['я', 'ты', 'он/она', 'мы', 'вы', 'они']
            for p in person_order:
                if p in merged:
                    print(f"   {p}: {merged[p]}")
            print()
    else:

        if forms.get('present'):
            print(" Настоящее время:")
            person_order = ['я', 'ты', 'он/она', 'мы', 'вы', 'они']
            for p in person_order:
                if p in forms['present']:
                    print(f"   {p}: {forms['present'][p]}")
            print()
        if forms.get('future_simple'):
            print(" Будущее время:")
            person_order = ['я', 'ты', 'он/она', 'мы', 'вы', 'они']
            for p in person_order:
                if p in forms['future_simple']:
                    print(f"   {p}: {forms['future_simple'][p]}")
            print()


    if forms.get('future_compound'):
        print(" Сложное будущее:")
        person_order = ['я', 'ты', 'он/она', 'мы', 'вы', 'они']
        for p in person_order:
            if p in forms['future_compound']:
                print(f"   {p}: {forms['future_compound'][p]}")
        print()




def main_loop():
    print("\n\033[1m\033[4mУмный морфологический анализатор/синтезатор глаголов (FST-aware)\033[0m\n")

    fst_proc = get_fst_processor()
    pym_proc = get_pymorphy_processor()
    mode = None

    while True:

        if mode is None:
            print("Выберите режим работы:")
            print(" 1 - Чистый FST (SimpleFST)")
            print(" 2 - FST + pymorphy3 (обученный автомат)")
            print(" 3 - Чистый pymorphy3")
            print(" q - Выход")
            sel = input("Ввод: ").strip().lower()
            if sel == "q":
                print("Выход из программы.")
                return
            if sel not in ("1", "2", "3"):
                print("Неверный выбор. Попробуйте снова.")
                continue
            mode = sel
            if mode == "1":
                print("\nРежим выбран: SimpleFST (чистый FST)\n")
            elif mode == "2":
                print("\nРежим выбран: FST+pymorphy3 (обученный автомат)\n")
                load_trained_fst(fst_proc, prefix="trained")
            else:
                print("\nРежим выбран: pymorphy3\n")


        print("Выберите действие:")
        print(" 1 - Синтез: инфинитив → все формы")
        print(" 2 - Анализ: форма → инфинитив + граммемы")
        print(" m - Сменить режим")
        print(" q - Выход")
        choice = input("Ввод: ").strip().lower()

        if choice == 'q':
            print("Выход.")
            break
        if choice == 'm':
            mode = None
            continue

        # ---------- СИНТЕЗ ----------
        if choice == "1":
            infinitive = input("Введите инфинитив: ").strip().lower()
            if not infinitive:
                print("Пустой ввод.")
                continue

            t0 = time.time()

            if mode == "1":
                try:
                    forms = fst_proc.fst.synthesize(infinitive)
                    source = forms.get("source", "simple-fst") if isinstance(forms, dict) else "simple-fst"
                except Exception:
                    forms = {}
                    source = "unknown"

            elif mode == "2":
                try:
                    forms = fst_proc.synthesize(infinitive)
                    source = forms.get("source", "fst+pymorphy3") if isinstance(forms, dict) else "fst+pymorphy3"
                except Exception:
                    try:
                        forms = pym_proc.synthesize(infinitive)
                        source = forms.get("source", "pymorphy3") if isinstance(forms, dict) else "pymorphy3"
                    except Exception:
                        forms = {}
                        source = "unknown"

            else:
                try:
                    forms = pym_proc.synthesize(infinitive)
                    source = forms.get("source", "pymorphy3") if isinstance(forms, dict) else "pymorphy3"
                except Exception:
                    forms = {}
                    source = "unknown"

            elapsed = time.time() - t0

            try:
                pretty_print_synth(forms, infinitive)
            except Exception:
                print("Ошибка при выводе форм.")
                print(forms)

            print(f"\n(Источник: {source.upper()})")
            print(f"Время выполнения: {elapsed:.4f} сек.")
            print("-" * 50)
            continue

        # ---------- АНАЛИЗ ----------
        if choice == "2":
            form = input("Введите форму глагола: ").strip().lower()
            if not form:
                print("Пустой ввод.")
                continue

            t0 = time.time()
            analysis = None
            source = None

            if mode == "1":
                try:
                    analysis = fst_proc.fst.analyze(form)
                    source = "simple-fst"
                except Exception:
                    analysis = None

            elif mode == "2":
                try:
                    analysis = fst_proc.analyze(form)
                    source = getattr(analysis, "source", None) or (
                        "fst" if getattr(fst_proc, "_fst_trained", False) else "pymorphy3"
                    )
                except Exception:
                    analysis = None
                if not analysis:
                    try:
                        analysis = pym_proc.analyze(form)
                        source = getattr(analysis, "source", "pymorphy3")
                    except Exception:
                        analysis = None
                if not analysis:
                    try:
                        analysis = fst_proc.fst.analyze(form)
                        source = "simple-fst"
                    except Exception:
                        analysis = None

            else:
                try:
                    analysis = pym_proc.analyze(form)
                    source = getattr(analysis, "source", "pymorphy3")
                except Exception:
                    analysis = None

            elapsed = time.time() - t0

            if not analysis:
                print(f"Форма '{form}' не распознана как глагол")
                print("-" * 50)
                continue

            print(f"\nАнализ формы '{form}':")

            if hasattr(analysis, 'infinitive'):
                print(f" Инфинитив: {analysis.infinitive}")
                atense = getattr(analysis, 'tense', None)
                aaspect = getattr(analysis, 'aspect', None)
                aperson_raw = getattr(analysis, 'person', None)
                agender = getattr(analysis, 'gender', None)
                anumber = getattr(analysis, 'number', None)
            else:
                print(f" Инфинитив: {analysis.get('инфинитив') or analysis.get('infinitive')}")
                time_str = (analysis.get('время') or analysis.get('time') or "").lower()
                if "прош" in time_str or "past" in time_str:
                    atense = 'past'
                elif "буд" in time_str or "future" in time_str:
                    atense = 'future'
                else:
                    atense = 'present'

                asp_raw = (analysis.get('аспект') or analysis.get('aspect') or "").lower()
                if 'несов' in asp_raw or 'impf' in asp_raw:
                    aaspect = 'impf'
                elif 'двув' in asp_raw or 'bia' in asp_raw:
                    aaspect = 'biaspectual'
                elif 'соверш' in asp_raw or 'perf' in asp_raw:
                    aaspect = 'perf'
                else:
                    aaspect = 'impf'

                aperson_raw = analysis.get('person') or analysis.get('лицо/число') or analysis.get('лицо')

                rn = analysis.get('род/число') or analysis.get('род') or analysis.get('gender')
                agender = None
                anumber = None
                if rn:
                    if rn in ('masc', 'femn', 'neut'):
                        agender = rn
                        anumber = 'sing'
                    elif rn == 'plur':
                        agender = None
                        anumber = 'plur'
                    else:
                        hr_map = {
                            'мужской род': ('masc', 'sing'),
                            'женский род': ('femn', 'sing'),
                            'средний род': ('neut', 'sing'),
                            'множественное число': (None, 'plur'),
                            'множественное число.': (None, 'plur')
                        }
                        if rn in hr_map:
                            agender, anumber = hr_map[rn]
                        else:
                            rn_low = rn.lower()
                            if 'муж' in rn_low:
                                agender, anumber = 'masc', 'sing'
                            elif 'жен' in rn_low:
                                agender, anumber = 'femn', 'sing'
                            elif 'средн' in rn_low:
                                agender, anumber = 'neut', 'sing'
                            elif 'мн' in rn_low or 'множе' in rn_low:
                                agender, anumber = None, 'plur'
                            else:
                                for code in ('masc', 'femn', 'neut', 'plur'):
                                    if code in rn_low:
                                        if code == 'plur':
                                            agender, anumber = None, 'plur'
                                        else:
                                            agender, anumber = code, 'sing'
                                        break

            # ---- вывод параметров ----
            if atense == 'future':
                if form.startswith(('буду ', 'будешь ', 'будет ', 'будем ', 'будете ', 'будут ')):
                    tense_name = 'Сложное будущее'
                else:
                    tense_name = 'Простое будущее'
            else:
                tense_name = {'past': 'Прошедшее', 'present': 'Настоящее'}.get(atense, 'Неизвестно')
            print(f" Время: {tense_name}")

            aspect_name = (
                'двувидовой' if aaspect == 'biaspectual'
                else ('совершенный' if aaspect == 'perf' else 'несовершенный')
            )
            print(f" Вид: {aspect_name}")

            if atense in ('present', 'future') and aperson_raw:
                person_map = {
                    '1sg': '1 лицо, ед. число', '2sg': '2 лицо, ед. число', '3sg': '3 лицо, ед. число',
                    '1pl': '1 лицо, мн. число', '2pl': '2 лицо, мн. число', '3pl': '3 лицо, мн. число'
                }
                pronoun_to_label = {
                    'я': '1 лицо, ед. число', 'ты': '2 лицо, ед. число', 'он/она': '3 лицо, ед. число',
                    'мы': '1 лицо, мн. число', 'вы': '2 лицо, мн. число', 'они': '3 лицо, мн. число'
                }
                person_display = None

                if isinstance(aperson_raw, str) and 'лицо' in aperson_raw:
                    person_display = aperson_raw
                elif isinstance(aperson_raw, str) and aperson_raw in person_map:
                    person_display = person_map[aperson_raw]
                elif isinstance(aperson_raw, str) and aperson_raw in pronoun_to_label:
                    person_display = pronoun_to_label[aperson_raw]
                else:
                    if isinstance(aperson_raw, str):
                        for key in person_map:
                            if key in aperson_raw:
                                person_display = person_map[key]
                                break
                if not person_display:
                    person_display = aperson_raw
                print(f" Лицо и число: {person_display}")

            if atense == 'past' and agender:
                gender_map = {'masc': 'мужской род', 'femn': 'женский род', 'neut': 'средний род'}
                print(f" Род: {gender_map.get(agender, agender)}")

            if atense == 'past' and anumber:
                number_name = 'единственное число' if anumber == 'sing' else 'множественное число'
                print(f" Число: {number_name}")

            print(f"\n(Источник: {str(source).upper()})")
            print(f"Время выполнения: {elapsed:.4f} сек.")
            print("-" * 50)
            continue

        print("Неверный выбор. Попробуйте снова.")


In [ ]:
fst_proc = get_fst_processor()
train_and_save_fst(fst_proc, limit=10000)

load_trained_fst(fst_proc)
main_loop()

 Найдено 185 глаголов (limit=10000), начинаем обучение FST...
 FST обучен на 2529 парах (время: 2.96s)
 FST сохранён в файлы trained_synth.fst и trained_analyze.fst.
 Trained FSTs loaded.

Умный морфологический анализатор/синтезатор глаголов (FST-aware)

Выберите режим работы:
 1 - Чистый FST (SimpleFST)
 2 - FST + pymorphy3 (обученный автомат)
 3 - Чистый pymorphy3
 q - Выход
Ввод: 1

Режим выбран: SimpleFST (чистый FST)

Выберите действие:
 1 - Синтез: инфинитив → все формы
 2 - Анализ: форма → инфинитив + граммемы
 m - Сменить режим
 q - Выход
Ввод: 1
Введите инфинитив: делать

 Формы глагола 'делать':

 Инфинитив: делать (несовершенный вид)

 Прошедшее время:
   мужской род: делал
   женский род: делала
   средний род: делало
   множественное число: делали

 Настоящее время:
   я: делаю
   ты: делаешь
   он/она: делает
   мы: делаем
   вы: делаете
   они: делают

 Сложное будущее:
   я: буду делать
   ты: будешь делать
   он/она: будет делать
   мы: будем делать
   вы: будете делат